Multi-year (interannual) capacity-factor variability for BC — CELL LEVEL.
----



## Rationale

Cluster assignments are recomputed independently for each weather year, so cluster
identities/boundaries are not comparable across years. Interannual variability is
therefore quantified on the FIXED grid of cells (directly comparable), not clusters.

For each cell and technology (solar, wind) this script computes, over all available
ERA5 weather years:
  - interannual mean CF, standard deviation, coefficient of variation (CV%)
  - interannual mean/sd/CV of the screening-level LCOE proxy
And at the regional level:
  - cell-level rank stability: Spearman rho of per-cell LCOE proxy in each year vs a
    reference year (cells matched by stable cell index), plus rho of CF ranking.

Outputs (written to vis/multiyear_era5/):
  - cell_interannual_cf_stats.csv     (per-cell statistics)
  - region_interannual_summary.csv    (province-level summary of cell CVs)
  - cell_rank_stability_spearman.csv  (per-year rho vs reference year, cell level)
  - fig_M4_cf_cv_distribution.png/.tiff
  - fig_M4_supply_curve_envelope.png/.tiff

Run once the per-year stores are fully downloaded (hydrated) locally:
    python multiyear_cf_variability.py
"""

## imports

In [ ]:
from __future__ import annotations
import re
import sys
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", category=UserWarning)

## Configuration

In [ ]:
# --------------------------------------------------------------------------- #
# Configuration
# --------------------------------------------------------------------------- #
SCENARIO = "BASELINE"          # multi-year variability is assessed on the baseline
COUNTRY_KWD = "Canada"
REGION_CODE = "BC"
REFERENCE_YEAR = 2024          # reference year for rank-stability comparison

# %%
# LCOE thresholds used elsewhere in the paper (USD/MWh)
SOLAR_LCOE_THRESHOLD = 90
WIND_LCOE_THRESHOLD = 130

# %%
# Resolve paths exactly as the notebook does: notebook lives in Publication_resources,
# repo root is two levels up; the store lives under <paper_resources>/data/store too.
PAPER_RESOURCES = Path.cwd()
ROOT = PAPER_RESOURCES.parent.parent

# %%
# Candidate store directories (try paper-local first, then repo root).
STORE_DIRS = [
    PAPER_RESOURCES / f"data/store/{COUNTRY_KWD}/{REGION_CODE}",
    ROOT / f"data/store/{COUNTRY_KWD}/{REGION_CODE}",
]

# %%
OUT_DIR = PAPER_RESOURCES / "vis" / "multiyear_era5"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# %%
# Capacity-factor and LCOE column names per technology (per the store schema).
TECHS = ("solar", "wind")
CF_COL = {t: f"{t}_CF_mean" for t in TECHS}
LCOE_COL = {t: f"lcoe_{t}" for t in TECHS}
CAP_COL = {t: f"potential_capacity_{t}" for t in TECHS}
LCOE_THRESH = {"solar": SOLAR_LCOE_THRESHOLD, "wind": WIND_LCOE_THRESHOLD}


## Load Stores

In [ ]:
# --------------------------------------------------------------------------- #
# Store loading
# --------------------------------------------------------------------------- #
def find_store_dir() -> Path:
    for d in STORE_DIRS:
        if d.exists() and any(d.glob(f"resources_{COUNTRY_KWD}_{REGION_CODE}_{SCENARIO}_*.h5")):
            return d
    raise FileNotFoundError(
        "No store directory with baseline .h5 files found. Looked in:\n  "
        + "\n  ".join(str(d) for d in STORE_DIRS)
    )

def discover_year_files(store_dir: Path) -> dict[int, Path]:
    """Map weather_year -> store path for the BASELINE scenario.

    Filenames look like: resources_Canada_BC_BASELINE_<YEAR>_<rundate>.h5
    The scenario token must be exactly BASELINE (excludes BASELINE_GRID_RESTRICTED etc.).
    If several run-dates exist for a year, the most recent is kept.
    """
    pat = re.compile(
        rf"^resources_{COUNTRY_KWD}_{REGION_CODE}_{SCENARIO}_(\d{{4}})_(\d+)\.h5$"
    )
    found: dict[int, tuple[int, Path]] = {}
    for p in store_dir.glob(f"resources_{COUNTRY_KWD}_{REGION_CODE}_{SCENARIO}_*.h5"):
        m = pat.match(p.name)
        if not m:
            continue  # skip scenario variants (NO_BUFFERS, GRID_RESTRICTED, strict_policy_*)
        year, rundate = int(m.group(1)), int(m.group(2))
        if year not in found or rundate > found[year][0]:
            found[year] = (rundate, p)
    return {y: v[1] for y, v in sorted(found.items())}


def load_cells(store_path: Path) -> pd.DataFrame:
    """Load the 'cells' table from a store, preferring RES.DataHandler, with a
    pandas/HDFStore fallback so the script runs without the RES package installed."""
    try:
        if str(ROOT) not in sys.path:
            sys.path.insert(0, str(ROOT))
        from RESource.hdf5_handler import DataHandler  # type: ignore
        return DataHandler(store_path, show_structure=False).from_store("cells")
    except Exception:
        # Fallback: pandas pytables. The 'cells' GeoDataFrame is stored as a frame;
        # we only need numeric CF / LCOE / capacity columns, so geometry is optional.
        return pd.read_hdf(store_path, "cells")




## Core computation

In [ ]:
# --------------------------------------------------------------------------- #
# Core computation
# --------------------------------------------------------------------------- #
def stack_years(year_files: dict[int, Path]) -> pd.DataFrame:
    """Return long-format frame: index (cell_id) x year, with CF/LCOE/cap per tech."""
    frames = []
    for year, path in year_files.items():
        cells = load_cells(path)
        keep = []
        for t in TECHS:
            keep += [c for c in (CF_COL[t], LCOE_COL[t], CAP_COL[t]) if c in cells.columns]
        sub = cells[keep].copy()
        sub["year"] = year
        sub["cell_id"] = cells.index  # stable grid index
        frames.append(sub)
        print(f"  loaded {year}: {len(sub):,} cells from {path.name}")
    return pd.concat(frames, ignore_index=True)

def per_cell_stats(long_df: pd.DataFrame) -> pd.DataFrame:
    """Interannual mean/sd/CV per cell for CF and LCOE, per technology."""
    out = []
    g = long_df.groupby("cell_id")
    n_years = long_df["year"].nunique()
    rec = pd.DataFrame(index=sorted(long_df["cell_id"].unique()))
    rec["n_years"] = g["year"].nunique()
    for t in TECHS:
        cf, lc = CF_COL[t], LCOE_COL[t]
        if cf in long_df.columns:
            rec[f"{t}_CF_mean"] = g[cf].mean()
            rec[f"{t}_CF_sd"] = g[cf].std(ddof=1)
            rec[f"{t}_CF_cv_pct"] = 100 * rec[f"{t}_CF_sd"] / rec[f"{t}_CF_mean"]
        if lc in long_df.columns:
            rec[f"{t}_lcoe_mean"] = g[lc].mean()
            rec[f"{t}_lcoe_sd"] = g[lc].std(ddof=1)
            rec[f"{t}_lcoe_cv_pct"] = 100 * rec[f"{t}_lcoe_sd"] / rec[f"{t}_lcoe_mean"]
    rec.index.name = "cell_id"
    rec.attrs["n_years"] = n_years
    return rec


def rank_stability(long_df: pd.DataFrame, ref_year: int) -> pd.DataFrame:
    """Cell-level Spearman rho of per-cell LCOE proxy (and CF) for each year vs ref_year.
    Cells are matched by stable cell_id; only cells present in both years are used."""
    from scipy.stats import spearmanr

    rows = []
    years = sorted(long_df["year"].unique())
    ref = ref_year if ref_year in years else years[len(years) // 2]
    for t in TECHS:
        lc, cf = LCOE_COL[t], CF_COL[t]
        ref_df = long_df[long_df["year"] == ref].set_index("cell_id")
        for y in years:
            cur = long_df[long_df["year"] == y].set_index("cell_id")
            common = ref_df.index.intersection(cur.index)
            row = {"tech": t, "ref_year": ref, "year": y, "n_cells": len(common)}
            if lc in long_df.columns and len(common) > 2:
                rho, p = spearmanr(ref_df.loc[common, lc], cur.loc[common, lc])
                row["lcoe_spearman"], row["lcoe_p"] = round(rho, 4), round(p, 6)
            if cf in long_df.columns and len(common) > 2:
                rho, p = spearmanr(ref_df.loc[common, cf], cur.loc[common, cf])
                row["cf_spearman"], row["cf_p"] = round(rho, 4), round(p, 6)
            rows.append(row)
    return pd.DataFrame(rows)

def region_summary(stats: pd.DataFrame) -> pd.DataFrame:
    """Province-level summary: distribution of per-cell CVs and accessible-CF stats."""
    rows = []
    for t in TECHS:
        cv = stats.get(f"{t}_CF_cv_pct")
        cfm = stats.get(f"{t}_CF_mean")
        if cv is None:
            continue
        rows.append({
            "tech": t,
            "n_cells": int(cv.notna().sum()),
            "n_years": stats.attrs.get("n_years"),
            "cell_CF_cv_pct_median": round(float(cv.median()), 3),
            "cell_CF_cv_pct_mean": round(float(cv.mean()), 3),
            "cell_CF_cv_pct_p90": round(float(cv.quantile(0.90)), 3),
            "cell_CF_cv_pct_max": round(float(cv.max()), 3),
            "cell_CF_mean_median": round(float(cfm.median()), 4),
        })
    return pd.DataFrame(rows)



## Plots

In [ ]:
# --------------------------------------------------------------------------- #
# Figures
# --------------------------------------------------------------------------- #
def make_figures(stats: pd.DataFrame, long_df: pd.DataFrame) -> None:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    style = ROOT / "RES" / "visual_styles" / "elsevier.mplstyle"
    if style.exists():
        try:
            plt.style.use(str(style))
        except Exception:
            pass
    color = {"wind": "#2B8CBE", "solar": "#FE9929"}

    # (1) Distribution of per-cell CF CV%
    fig, ax = plt.subplots(figsize=(7, 4.5), dpi=300)
    for t in TECHS:
        col = stats.get(f"{t}_CF_cv_pct")
        if col is None:
            continue
        vals = col.dropna().values
        ax.hist(vals, bins=40, alpha=0.55, color=color[t], label=f"{t.title()} (n={len(vals)})")
    ax.set_xlabel("Per-cell interannual CF coefficient of variation (%)")
    ax.set_ylabel("Number of cells")
    ax.set_title(f"Interannual CF variability across cells — BC ({stats.attrs.get('n_years')} ERA5 years)")
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "fig_M4_cf_cv_distribution.jpg", dpi=300, bbox_inches="tight")
    plt.close(fig)

    # (2) Supply-curve envelope: per-year accessible-capacity supply curves overlaid
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=300)
    for ax, t in zip(axes, TECHS):
        lc, cap = LCOE_COL[t], CAP_COL[t]
        if lc not in long_df.columns or cap not in long_df.columns:
            continue
        for y in sorted(long_df["year"].unique()):
            d = long_df[(long_df["year"] == y)][[lc, cap]].dropna().sort_values(lc)
            d = d[d[lc] <= LCOE_THRESH[t]]
            if d.empty:
                continue
            x = np.cumsum(d[cap].values) / 1e3  # GW
            ax.step(x, d[lc].values, where="post", lw=0.8, alpha=0.6, label=str(y))
        ax.axhline(LCOE_THRESH[t], ls="--", lw=0.8, color="grey")
        ax.set_xlabel("Cumulative accessible capacity (GW)")
        ax.set_ylabel("Screening-level LCOE proxy (USD/MWh)")
        ax.set_title(f"{t.title()} supply-curve envelope")
        ax.legend(frameon=False, fontsize=7, ncol=2)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "fig_M4_supply_curve_envelope.jpg", dpi=300, bbox_inches="tight")
    # fig.savefig(OUT_DIR / "fig_M4_supply_curve_envelope.tiff", dpi=300, bbox_inches="tight")
    plt.close(fig)


## Main

In [ ]:
# --------------------------------------------------------------------------- #
# Main
# --------------------------------------------------------------------------- #
from multiyear_extended_plots import make_extended_figures
# def main() -> None:
store_dir = find_store_dir()
year_files = discover_year_files(store_dir)
if len(year_files) < 2:
    raise RuntimeError(f"Need >= 2 weather years; found {sorted(year_files)}")
print(f"Store dir : {store_dir}")
print(f"Years     : {sorted(year_files)}  (reference = {REFERENCE_YEAR})")

long_df = stack_years(year_files)
stats = per_cell_stats(long_df)
region = region_summary(stats)
ranks = rank_stability(long_df, REFERENCE_YEAR)

stats.to_csv(OUT_DIR / "cell_interannual_cf_stats.csv")
region.to_csv(OUT_DIR / "region_interannual_summary.csv", index=False)
ranks.to_csv(OUT_DIR / "cell_rank_stability_spearman.csv", index=False)
make_figures(stats, long_df)

print("\n=== Region-level summary (per-cell CF CV%) ===")
print(region.to_string(index=False))
print(f"\nOutputs written to: {OUT_DIR}")
# Sanity guard against the earlier zero-variance artefact:
for t in TECHS:
    col = stats.get(f"{t}_CF_cv_pct")
    if col is not None and np.nan_to_num(col.values).max() == 0:
        print(f"WARNING: {t} CF CV is identically zero across all cells — "
                f"check that distinct weather years were loaded (possible single-year propagation).")


make_extended_figures(stats, long_df, ranks, LCOE_THRESH, OUT_DIR)

# Run Main

In [ ]:
# # %%
# if __name__ == "__main__":
#     main()

    